In [ ]:
# reload modules on import
%load_ext autoreload
%autoreload 2
# imports
import numpy as np

## Tuning the regressorlens
The following notebook is concerned with finding good hyperparameters for the regressorlens.
We will focus on R2 performance on the canonical validation set defined in the data loader. This includes models from earlt, middle and final stages of training in equal proportions

First let's load in the data

In [ ]:
from cnn_surgery.utils.load_dataset import load_dataset
early = load_dataset('mnist', metrics_file='metrics_merged_mnist_early.csv', load_class_acc=True, stage='early')
middle = load_dataset('mnist', metrics_file='metrics_merged_mnist_middle.csv', load_class_acc=True, stage='middle')
final = load_dataset('mnist', metrics_file='metrics_merged.csv', load_class_acc=True, stage='final')

train_early, test_early, val_early = early
train_middle, test_middle, val_middle = middle
train_final, test_final, val_final = final

weights_train = np.concatenate([train_early[0], train_middle[0], train_final[0]])
weights_val = np.concatenate([val_early[0], val_middle[0], val_final[0]])

accuracies_train = np.concatenate([train_early[1][:, -10:], train_middle[1][:, -10:], train_final[1][:, -10:]])
accuracies_val = np.concatenate([val_early[1][:, -10:], val_middle[1][:, -10:], val_final[1][:, -10:]])

In [ ]:
print(f"Train set size: {len(weights_train)}, Validation set size: {len(weights_val)}")

Let's have a look at the default config

In [ ]:
from cnn_surgery.lenses.regressor_lens import get_regressor_lens, default_config
config = default_config.copy()
config

I think it makes sense to stick with Adam since everybody uses that nowadays. For initialisation, I do not see a reason not to stick with pytorch standard kaiming / he normal.

What we probably should tune: (in order of importance)
* Batch Size
* Learning Rate
* Architecture (depth / width)
* L2 Penalty
* Dropout rate

Using results from Unterthiner as an educated guess. These came from a hyperparameter tuning effort so they are oddly specific

In [ ]:
config['early_stopping'] = True  # enable early stopping

In [ ]:
# T for mps: 2.99s, 3.12s
get_regressor_lens(weights_train, accuracies_train, weights_val, accuracies_val, verbose=True, config=config, return_metrics=True)

In [ ]:
config['learning_rate'] = 5e-4
config['w_init_name'] = 'kaiming'
config['n_layers'] = 5
config['n_hiddens'] = 1024

In [ ]:
config